In [ ]:
# Use with `Bash (micromamba)`
# Sync data files for streamlit app
SRC='euler.ethz.ch:/cluster/home/jjaenes/af2genomics/results/human-mutfunc-data/'
DEST='./data/'
ARGS='--progress '
#ARGS+='--delete '
#ARGS+='--dry-run '
rsync -azv $ARGS "$SRC" "$DEST" \
    --include='/missense.parquet' \
    --include='/monomers-db' \
    --include='/monomers-db.dbtype' \
    --include='/monomers-db.index' \
    --include='/monomers-db.lookup' \
    --include='/multimers-db' \
    --include='/multimers-db.dbtype' \
    --include='/multimers-db.index' \
    --include='/multimers-db.lookup' \
    --include='/multimers.parquet' \
    --include='/pockets.parquet' \
    --exclude='*'
du -hs data/

In [ ]:
# Generate tag/version
BUILD="$(date +%y.%m)-$(git rev-parse --short HEAD)"
if ! git diff --quiet && git diff --cached --quiet; then
    BUILD+="-dirty"
fi
echo $BUILD

In [ ]:
# Build container image with /data included
ARGS=(
    --progress plain
    --build-arg MUTFUNC_BUILD=$BUILD
    --platform=linux/amd64
    #--platform=linux/arm64,linux/amd64
    --tag mutfunc:$BUILD
)
docker buildx build "${ARGS[@]}" .

In [ ]:
# Quick test - emulation so will be slow..
docker run -p 8501:8501 mutfunc:$BUILD

In [ ]:
# Save image as tar archive that can be copied over to the host VM
docker save mutfunc:$BUILD -o mutfunc-$BUILD.tar

In [ ]:
# Copying image tar archive to host VM
SRC='./'
DEST='sysbc-lx-s04.ethz.ch:/images'
ARGS='--progress '
#ARGS+='--delete '
#ARGS+='--dry-run '
rsync -azv $ARGS "$SRC" "$DEST" \
    --include="mutfunc-$BUILD.tar" \
    --exclude='*'

In [ ]:
# On host VM, load image from tar archive - https://docs.docker.com/reference/cli/docker/image/load/
# $ ssh sysbc-lx-s04.ethz.ch
echo $ sudo docker load --input /images/mutfunc-$BUILD.tar

In [ ]:
# On host VM, stop old image and start new one
# $ sudo docker ps
# $ sudo docker stop <container_name_or_id>
echo $ sudo docker run -d -p 8501:8501 mutfunc:$BUILD

In [ ]:
# Clean up local images if needd
# $ docker system prune --all --volumes
docker images